In [ ]:
pip install openai python-dotenv pandas tqdm

In [ ]:
import os
import json
import re
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

project_root = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\FINAL\FINAL_STORAL_prompting")

raw_text_root = project_root / "raw_texts"
llm_output_root = project_root / "llm_outputs"
llm_output_root.mkdir(exist_ok=True)


final_storal = ["STORAL_001", "STORAL_002", "STORAL_003", "STORAL_005", "STORAL_006", "STORAL_007", "STORAL_008", "STORAL_010", "STORAL_011", "STORAL_012", "STORAL_014", "STORAL_015", "STORAL_016", "STORAL_017", "STORAL_018", "STORAL_020",]
# optimization_storal = ["STORAL_004", "STORAL_009", "STORAL_013", "STORAL_019"]

def read_storal_text(document_id):
    with open(raw_text_root / f"{document_id}.txt", "r", encoding="utf-8") as f:
        return f.read()

In [ ]:
client = OpenAI(
    api_key=# API KEY HERE,
)

In [ ]:
models = [
    "mistralai/mistral-small-3.2-24b-instruct",
    "qwen/qwen3.6-27b", 
    "google/gemma-4-31b-it"
]

model_file_names = {
    "mistralai/mistral-small-3.2-24b-instruct": "mistral-small-3.2-24b-instruct",
    "qwen/qwen3.6-27b": "qwen3.6-27b",
    "google/gemma-4-31b-it": "gemma-4-31b-it"
}


# read storal text


In [ ]:
def read_storal_text(document_id):
    path = raw_text_root / f"{document_id}.txt"

    with open(path, "r", encoding="utf-8") as f:
        return f.read()

# FINAL STORAL PROMPT: used on the other 16 STORAL stories, and the examples used are from developmental stories

In [ ]:
system_prompt_implicit_final = """
You are an expert annotator trained in implicit argumentation mining.

Your task is to identify implicit argumentative components in texts.

Implicit components are NOT directly stated in the text.
They must be inferred from context.

Return ONLY valid JSON.
Do not explain your reasoning.
Do not use markdown.
"""
def build_implicit_prompt_final(storal_text):

    return f"""

TASK:
Identify implicit argumentative components in the story text.

An implicit component is: NOT explicitly stated, but strongly implied by the text, necessary to understand the argumentative meaning, and supported by the text as a whole.

For every implicit component you MUST provide:
- label
- supporting span
- explicit formulation

IMPORTANT:
The supporting span is NOT the implicit component itself.
The supporting span is the text segment from which the implicit meaning is inferred.

Use ONLY these labels based on their definitions:

MC-IMP: Implicit major claim is the overall inferred takeaway, moral, lesson, central message, or broad conclusion of the text.
C-IMP: Implicit claim: is an inferred intermediate viewpoint, interpretation, evaluative meaning, or unstated argumentative step.
P-IMP: Implicit premise is an unstated reasoning step, assumption, or inferred justification needed to connect ideas.

STRICT ANNOTATION RULES:

Only annotate implications that are STRONGLY supported.
Do NOT annotate speculative interpretations.
Do NOT invent morals unsupported by the text.
Do NOT generate multiple possible interpretations.
Select ONLY the best-supported interpretation.
The explicit formulation must stay close to the text.
The explicit formulation must be generalizable.
The explicit formulation must be a COMPLETE sentence.
The explicit formulation must NOT simply restate the supporting span.
If the inferred meaning is weak or ambiguous, do NOT annotate it.
If unsure, do NOT annotate.
Precision is MUCH more important than recall.

SUPPORTING SPAN RULES:

Copy EXACT text from the TED Talk.
Do NOT paraphrase the supporting span.
Select the MINIMAL span necessary for the inference.
Do NOT select the entire TED Talk.
Do NOT include unrelated surrounding text.
The supporting span should directly motivate the inferred meaning.

EXPLICIT FORMULATION RULES:

GOOD:

"Selfish behavior can harm others."
"Urban neglect damages public life."
"Experiencing consequences can teach responsibility."

BAD:

vague statements,
unsupported morals,
overly philosophical interpretations,
interpretations adding new ideas not grounded in the text.

NEGATIVE EXAMPLES:

Example 1

Text:
"It was raining heavily that morning."

Output:
{{
"annotations": []
}}

Example 2

Text:
"Thank you very much for inviting me here today."

Output:
{{
"annotations": []
}}

Example 3

Text:
"The bridge was built in 1952."

Output:
{{
"annotations": []
}}

POSITIVE EXAMPLES:

Example 4

Text:
"The hen and the grasshopper followed the advice of the dog and at the end of the year they had the necessary money for the cow. They also make a better  living through their saving."

Output:
{{
  "annotations": [
    {{
      "label": "P-IMP",
      "supporting_span": "The hen and the grasshopper followed the advice of the dog",
      "explicit_formulation": "You follow advice from a wise individual."
    }},
    {{
      "label": "MC-IMP",
      "supporting_span": "at the end of the year they had the necessary money for the cow. They also make a better  living through their saving",
      "explicit_formulation": "Hard work, responsible behavior and perseverance eventually leads to a better life."
    }}
  ]
}}

Example 5

Text:
"rather, you can just have a piece of leather cut in appropriate shape to cover your feet? The king was very much surprised by his suggestion and applauded the minister. He ordered for a pair of leather shoes for him and requested the countrymen to wear shoes."
Output:
{{
  "annotations": [
    {{
      "label": "MC-IMP",
      "supporting_span": "rather, you can just have a piece of leather cut in appropriate shape to cover your feet",
      "explicit_formulation": "It is easier to adapt than to try to change the whole world"
    }},
    {{
      "label": "P-IMP",
      "supporting_span": "The king was very much surprised by his suggestion and applauded the minister. He ordered for a pair of leather shoes for him and requested the countrymen to wear shoes",
      "explicit_formulation": "You find the simplest of ideas that turns out to be the most effective and least harmful"
    }}
  ]
}}

Example 6

Text:
""Now you understand?? are you sorry that because of your selfishness I got beaten by farmer.."Fox cried, "Yes.. yes.. I have realized my mistake.. now please forgive me..""

Output:
{{
  "annotations": [
    {{
      "label": "MC-IMP",
      "supporting_span": ""Now you understand?? are you sorry that because of your selfishness I got beaten by farmer.."",
      "explicit_formulation": "selfish actions can eventually bring consequences upon oneself"
    }},
    {{
      "label": "P-IMP",
      "supporting_span": "Yes.. yes.. I have realized my mistake.. now please forgive me..",
      "explicit_formulation": "You get a taste of your own medcine and it teaches you an important lesson"
    }}
  ]
}}

A text may contain:
- only one implicit major claim

OUTPUT FORMAT:
{{
  "annotations": [
    {{
      "label": "MC-IMP",
      "supporting_span": "exact copied text span",
      "explicit_formulation": "clear inferred statement"
    }}
  ]
}}


Story:
\"\"\"
{storal_text}
\"\"\"
"""

# clean JSON output

In [ ]:
def clean_llm_json_output(output):
    output = output.strip()

    output = re.sub(r"```json", "", output, flags=re.IGNORECASE)
    output = re.sub(r"```", "", output)

    start = output.find("{")
    end = output.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in model output.")

    output = output[start:end+1]

    parsed = json.loads(output)

    return json.dumps(parsed, ensure_ascii=False, indent=2)

# call openrouter

In [ ]:
def run_openrouter_prompt(model, system_prompt, user_prompt, temperature=0):
    kwargs = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature
    }

    # Sara said this may help for Gemma 4
    if model == "google/gemma-4-31b-it":
        kwargs["extra_body"] = {"reasoning": {"enabled": True}}

    completion = client.chat.completions.create(**kwargs)
    return completion.choices[0].message.content

# run all MODELS + all 16 STORAL

In [ ]:
prompt_version = "final"          
task_type = "implicit"

def run_one_job(document_id, model):
    storal_text = read_storal_text(document_id)

    safe_model_name = model_file_names[model]

    output_filename = (
        f"{document_id}_{safe_model_name}_"
        f"prompt-{prompt_version}_{task_type}.json"
    )

    output_path = llm_output_root / output_filename
    raw_error_path = llm_output_root / output_filename.replace(".json", "_RAW_ERROR.txt")

    raw_output = run_openrouter_prompt(
        model=model,
        system_prompt=system_prompt_implicit_final,
        user_prompt=build_implicit_prompt_final(storal_text),
        temperature=0
    )

    try:
        cleaned_output = clean_llm_json_output(raw_output)

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(cleaned_output)

        status = "success"

    except Exception as e:
        with open(raw_error_path, "w", encoding="utf-8") as f:
            f.write(raw_output)

        status = f"json_cleaning_error: {e}"

    return {
        "document": document_id,
        "model": safe_model_name,
        "prompt_version": prompt_version,
        "task_type": task_type,
        "output_file": str(output_path),
        "status": status
    }


jobs = []

for document_id in final_storal:
    for model in models:
        jobs.append((document_id, model))

run_log = []

with ThreadPoolExecutor(max_workers=1) as executor:
    futures = [
        executor.submit(run_one_job, document_id, model)
        for document_id, model in jobs
    ]

    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            run_log.append(future.result())
        except Exception as e:
            run_log.append({
                "document": None,
                "model": None,
                "prompt_version": prompt_version,
                "task_type": task_type,
                "output_file": None,
                "status": f"api_error: {e}"
            })

run_log_df = pd.DataFrame(run_log)
run_log_df

In [ ]:
failed = run_log_df[
    run_log_df["status"].str.contains("error", case=False, na=False)
]

for i, row in failed.iterrows():
    print("\n====================")
    print("MODEL:", row["model"])
    print("STATUS:")
    print(row["status"])